### **Resumen ejecutivo**
1. Estado de la Operación y Eficiencia Logística

Volumen y Cumplimiento: Se procesaron 99,441 órdenes, logrando una tasa de entrega efectiva del 97% (96,478 órdenes entregadas). El 8.11% de las entregas totales presentan problemas de puntualidad, lo que representa una oportunidad de mejora directa para elevar la satisfacción del cliente.

Foco Geográfico (Sao Paulo): Con 40,501 órdenes, el estado de SP concentra el mayor flujo operativo. La estrategia debe orientarse a la optimización de costos logísticos en esta zona para maximizar el margen de ganancia, dada su escala.

Puntos Críticos: Se identifica al estado de Alagoas (AL) como una zona de alto riesgo logístico, con un promedio de 24.5 días de entrega y una tasa de tardanza del 23%. Se recomienda una auditoría de costos de última milla para determinar si la operación en este estado es rentable o requiere un cambio de proveedor logístico.

2. Inteligencia Comercial y de Pagos

Perfil de Pago: Existe una alta dependencia del pago con tarjeta de crédito (76,505 órdenes), con un promedio de 4 cuotas por transacción. Esta tendencia es un activo valioso para el departamento de Marketing, permitiendo diseñar campañas que resalten las facilidades de financiación como diferenciador frente a la competencia.

Rentabilidad: Con un ticket promedio de 161, la empresa mantiene un modelo de negocio estable, pero con márgenes de mejora si se logra reducir la fricción en la entrega de órdenes tardías, las cuales impactan directamente en el flujo de caja y la lealtad del cliente.

3. Nota Metodológica

El análisis fue realizado mediante una arquitectura híbrida de datos. Se validó que, mientras Pandas es ideal para exploración ágil y manipulación de DataFrames, SQL (a través de JOIN y CASE WHEN) ofrece una mayor capacidad de síntesis y legibilidad para el cálculo de métricas complejas de negocio, optimizando el tiempo de procesamiento.



### **CONEXIÓN INICIAL**

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')

df_orders   = pd.read_csv('olist_orders_dataset.csv')
df_payments = pd.read_csv('olist_order_payments_dataset.csv')
df_customers = pd.read_csv('olist_customers_dataset.csv')

df_orders.to_sql('orders', conn, index=False, if_exists='replace')
df_payments.to_sql('payments', conn, index=False, if_exists='replace')
df_customers.to_sql('customers', conn, index=False, if_exists='replace')

99441

###Top 5 Estados por Volumen (Consulta A)

In [2]:
# Pregunta de negocio: ¿Cuáles son los 5 estados con mayor volumen de pedidos entregados?
query_top_estados = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) as total_ordenes
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_ordenes DESC
LIMIT 5;
"""
df_top_5 = pd.read_sql_query(query_top_estados, conn)
display(df_top_5)

,customer_state,total_ordenes
0,SP,40501
1,RJ,12350
2,MG,11354
3,RS,5345
4,PR,4923


###Insight Identificado

Tal y como se evidenciaba se observa que en el estado de SAO PAULO es el que más predomina con exactamente 40501 ordenes que indican la gran cantidad de flujo de información, lo que sugiere que focalicemos nuestros esfuerzos en minimizar el costo logistico de Sao Paulo, para así obtener una ventaja competitiva.

### Análisis de eficiencia y Tardanza

In [3]:
# Pregunta de negocio: ¿Cuál es el tiempo promedio de entrega y el % de órdenes críticas por estado?
query_eficiencia = """
SELECT
    c.customer_state,
    AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)) as avg_dias_entrega,
    SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as pct_tardanza
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY pct_tardanza DESC;
"""
df_metricas_sql = pd.read_sql_query(query_eficiencia, conn)
display(df_metricas_sql.head(10))

,customer_state,avg_dias_entrega,pct_tardanza
0,AL,24.543855,23.929471
1,MA,21.572976,19.665272
2,PI,19.457098,15.966387
3,CE,21.266579,15.324472
4,SE,21.519788,15.223881
5,BA,19.335466,14.035627
6,RJ,15.309438,13.473684
7,TO,17.658063,12.773723
8,PA,23.772917,12.367865
9,ES,15.789307,12.230576


 ### Insights Encontrados

 Se observa que el tiempo promedio de entrega y el porcentaje de tardanza por estado en donde se identifica el estado de Alagoas con un promedio de 24.5 días con un porcentaje de tardanza de 23%, lo cual sugiere una investigación de los costos que implican esas tardanzas, así como tambien su impacto en las ordenes adquiridas.
 Si el costo de enviar a AL es mayor que el margen de beneficio de esas ventas, quizás la empresa deba dejar de operar en ese estado o cambiar de transportista

### **Preguntas de negocio**


In [23]:
#Nivel 1 Diagnostico-Ticket promedio
Diagnostico_General= """
SELECT COUNT(order_id) as total_ordenes,
       SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) as entregadas
FROM orders"""
df_diagnostic_sql = pd.read_sql_query(Diagnostico_General, conn)
display(df_diagnostic_sql)
Ticket_promedio_orden= """
SELECT AVG(total_pago) as ticket_promedio
FROM (SELECT order_id, SUM(payment_value) as total_pago FROM payments GROUP BY order_id) """
df_promedioorden_sql = pd.read_sql_query(Ticket_promedio_orden, conn)
display(df_promedioorden_sql)

Porcentaje_Tarde ="""
SELECT
    SUM(CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*) as pct_tardanza
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL;
"""
df_porcentaje_sql = pd.read_sql_query(Porcentaje_Tarde, conn)
display(df_porcentaje_sql)

promedio_cuotas = """
SELECT AVG(payment_installments) as promedio_cuotas
FROM payments WHERE payment_type = 'credit_card';
"""
df_promediocuotas_sql = pd.read_sql_query(promedio_cuotas, conn)
display(df_promediocuotas_sql)

Órdenes_método_de_pago = """
SELECT payment_type, COUNT(DISTINCT order_id) as total_ordenes
FROM payments GROUP BY payment_type;
"""
df_metodoPago_sql = pd.read_sql_query(Órdenes_método_de_pago, conn)
display(df_metodoPago_sql)






,total_ordenes,entregadas
0,99441,96478


,ticket_promedio
0,160.990267


,pct_tardanza
0,8.112367


,promedio_cuotas
0,3.507155


,payment_type,total_ordenes
0,boleto,19784
1,credit_card,76505
2,debit_card,1528
3,not_defined,3
4,voucher,3866


### Insight identificada
Se observa que en total hubo 99441 ordenes, de las cuales se entregarón 96478 con ticket promedio de 161, con un porcentaje de tardanza general de 8.11%, los cuales se evidencian en las casi 2963 ordenes que no se entregarón, lo cual sugiere, que se necesitan acciones para la mejora continua de este indicador.
Por otro lado se encontró que el promedio de cuotas es de casi 4 cuotas, está información es útil para las campañas de marketing que tenga la empresa a futuro. En cuanto a las ordenes por metodo de pago, se evidencia lo que se encontró anteriormente en donde predomina la tarjeta de credito con un total de ordenes de 76505


### **Comparación con Pandas**

En Pandas: Para obtener el estado de un pedido, usamos pd.merge(df_orders, df_customers, on='customer_id').

En SQL: Utilizamos la cláusula JOIN customers c ON o.customer_id = c.customer_id.

Observación: SQL resulta más legible para realizar agregaciones complejas y filtros condicionales (CASE WHEN) en una sola sentencia, mientras que en Pandas requeriría varias líneas de código y creación de columnas temporales.


In [22]:
# Definimos el contenido del archivo .sql como una cadena de texto
queries_sql = """
-- NIVEL 1: Diagnóstico General
-- 1.1 Total órdenes y órdenes entregadas
SELECT COUNT(order_id) as total_ordenes,
       SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) as entregadas
FROM orders;

-- 1.2 Ticket promedio por orden
SELECT AVG(total_pago) as ticket_promedio
FROM (SELECT order_id, SUM(payment_value) as total_pago FROM payments GROUP BY order_id);

-- 1.3 Top 5 estados con más órdenes entregadas
SELECT c.customer_state, COUNT(DISTINCT o.order_id) as total_ordenes
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_ordenes DESC
LIMIT 5;

-- NIVEL 2: Eficiencia Operativa
-- 2.1 Días promedio de entrega
SELECT AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)) as promedio_dias
FROM orders WHERE order_status = 'delivered';

-- 2.2 Porcentaje de órdenes tarde
SELECT SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as pct_tardanza
FROM orders WHERE order_status = 'delivered';

-- NIVEL 3: Comportamiento de Pago
-- 3.1 Órdenes por método de pago
SELECT payment_type, COUNT(DISTINCT order_id) as total_ordenes
FROM payments GROUP BY payment_type;

-- 3.2 Promedio de cuotas en tarjeta de crédito
SELECT AVG(payment_installments) as promedio_cuotas
FROM payments WHERE payment_type = 'credit_card';
"""

# Guardar en un archivo
with open('analisis_olist.sql', 'w') as f:
    f.write(queries_sql)

print("Archivo 'analisis_olist.sql' creado con éxito. Puedes descargarlo desde la pestaña de archivos de la izquierda.")

Archivo 'analisis_olist.sql' creado con éxito. Puedes descargarlo desde la pestaña de archivos de la izquierda.
